# Live risk testing: pull all trades via Quantifi DSP
Read-only: lists every trade with HQL, then builds pricers and computes PV.

In [1]:
import pandas as pd
import os
import time

# Toolkit import so the pricer objects returned by DSP can be used
from Quantifi.Toolkit.Base import Dt

# Quantifi DSP REST client
from qservices.QuantifiApiClient import *

In [ ]:
# Environment: switch between Prod and UAT here
ENV = 'Prod'   # 'Prod' or 'UAT'
host = {'Prod': 'hb-dc1pd-qfidsp', 'UAT': 'hb-dc1ut-qfidsp'}[ENV]
os.environ["Q_DATA_SERVICE_URL_PREFIX"] = f"https://{host}:47200"
os.environ["Q_MARKET_ENV_SERVICE_URL_PREFIX"] = f"https://{host}:47201"
os.environ["Q_LOCAL_INSTALL"] = "True"

requests.packages.urllib3.disable_warnings()   # unverified HTTPS cert

AS_OF = '2026-11-16'      # pricing date
CALC_ENV = 'Official'
MAX_TRADES = 100_000       # data_query silently truncates at this many rows (default is 500)

## 1. List all trades

In [3]:
# Only Abhishek Bhutra's trades. He appears in the DB under several Trader.Name values
# (AB = Abhishek, AC = Aman Chind is excluded).
TRADERS = ['abhutra', 'AB', 'Abhishek Bhutra']
trader_list = ', '.join(f"'{t}'" for t in TRADERS)

all_trades = Connect.data_query(f"FROM Trade b WHERE b.Trader.Name IN ({trader_list})",
                                max_results=MAX_TRADES, verify_https_cert=False)
print('Trades returned:', len(all_trades))
if len(all_trades) >= MAX_TRADES:
    print('WARNING: hit MAX_TRADES, results are probably truncated; raise it.')

import pricer_cache as pc

# Scope: only Credit (official booking) and Manual strategies, only trades live at AS_OF
# (Done, traded on/before, not terminated/matured/expired). Add 'TradeIdeas' etc. here later.
STRATEGIES = ['Credit', 'Manual']
trade_ids = pc.live_trade_ids(all_trades, AS_OF, include_strategies=STRATEGIES)
print(f'{len(trade_ids)} live trades at {AS_OF} in strategies {STRATEGIES} (of {len(all_trades)} for these traders)')
print('Keys on a trade record:', sorted(all_trades[0].keys()) if all_trades else None)

Trades returned: 9730
Keys on a trade record: ['$type', 'Amount', 'Counterparty', 'Groups', 'InitialMargin', 'IsCleared', 'Notes', 'ObjectId', 'Payment', 'PaymentCcy', 'PaymentSettle', 'Product', 'Settle', 'Strategy', 'SubStrategy', 'Tags', 'Termination', 'TradeId', 'TradeStatus', 'TradeType', 'Traded', 'TradedLevel', 'TradedLevelType', 'Trader']


In [4]:
# Flat static view of the trades (top-level scalar fields only)
trades_static = pd.json_normalize(all_trades, max_level=1)


def flatten_tags(df, col, prefix):
    """Turn a column of [{'Name':..., 'Value':...}, ...] into one column per tag name."""
    def to_dict(tags):
        if not isinstance(tags, list):
            return {}
        return {f"{prefix}{t['Name']}": t.get('Value') for t in tags if isinstance(t, dict) and 'Name' in t}
    wide = pd.DataFrame([to_dict(t) for t in df[col]], index=df.index)
    return pd.concat([df.drop(columns=col), wide], axis=1)


for col, prefix in [('Tags', 'Tag.'), ('Product.Tags', 'Product.Tag.')]:
    if col in trades_static:
        trades_static = flatten_tags(trades_static, col, prefix)

# Most recent trades first (parse to datetime for sorting; unparseable/missing dates go last)
if 'Traded' in trades_static:
    trades_static = (trades_static
                     .sort_values('Traded', ascending=False, na_position='last',
                                  key=lambda s: pd.to_datetime(s, errors='coerce', utc=True))
                     .reset_index(drop=True))

# Show every column (no "..." in the middle) and don't truncate cell text
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None,
                       'display.width', None):
    display(trades_static.head(5))
print(f'{trades_static.shape[1]} columns; trader column(s):',
      [c for c in trades_static.columns if 'trader' in c.lower()])
trades_static.to_csv('all_trades_static.csv', index=False)

,$type,ObjectId,TradeId,TradeStatus,TradeType,Traded,Settle,Termination,Amount,Payment,PaymentSettle,TradedLevel,Notes,IsCleared,InitialMargin,PaymentCcy,Groups,TradedLevelType.$type,TradedLevelType.Name,TradedLevelType.ObjectId,Strategy.$type,Strategy.Name,Strategy.ObjectId,SubStrategy.$type,SubStrategy.Name,SubStrategy.ObjectId,Trader.$type,Trader.Name,Trader.ObjectId,Counterparty.$type,Counterparty.Name,Counterparty.ObjectId,Product.$type,Product.Name,Product.ObjectId,BookingEntity.$type,BookingEntity.Name,BookingEntity.ObjectId,ClearingBroker.$type,ClearingBroker.Name,ClearingBroker.ObjectId,Product.Description,Product.Effective,Product.Maturity,Product.DaysToSettle,Product.Ccy,Product.Notional,Product.ClearingBroker,Product.Premium,Product.DayCount,Product.Frequency,Product.Calendar,Product.BDConvention,Product.AccruedOnDefault,Product.PayRecoveryAtMaturity,Product.FixedRecovery,Product.FixedRecoveryRate,Product.PremiumInAdvance,Product.Seniority,Product.ReferenceEntity,Product.ReferenceObligation,Product.SettlementMethod,Product.PhysicalCreditSettlement,Product.CashCreditSettlement,Product.CreditEvents,Product.Obligation,Product.DeliverableObligation,Product.CalcAgentInfo,Product.CdsType,Product.Resets,Product.CycleRule,Product.AmortizationSchedule,Product.CouponSchedule,Product.BonusPremium,Product.ReferenceObligationOption,Product.TransactionType,Product.ISDADefinition,TradeDocumentation.$type,TradeDocumentation.ObjectId,TradeDocumentation.Documentation,TradeDocumentation.CalcAgentAsSpecifiedInMaster,TradeDocumentation.DocumentationType,TradeDocumentation.MasterAgreementType,Product.ExternalId,Product.CreditEvent,Product.CdoType,Product.Underlyings,Product.SharedBasket,Product.Attachment,Product.Detachment,Product.Freq,Product.AmortizePremium,Product.FeeGuaranteed,ClearingHouse.$type,ClearingHouse.Name,ClearingHouse.ObjectId,LeadTrade.$type,LeadTrade.TradeId,LeadTrade.ObjectId,Product.Underlying,Product.Expiration,Product.Type,Product.Style,Product.Strike,Product.SettlementType,Product.TrancheType,Product.CDSIndex,Tag.DealCodeDescription,Tag.DealCodeId,Tag.SecurityId,Tag.StrategyId,Tag.UniqueId,Tag.AssociatedCreditIndex
0,CDXTrancheTrade,256423703784094720,CDXTranche0024666,Done,New,2026-11-16,2026-11-16,NaN,25000000.0,0.00,NaN,0.00000,[],False,0.0,USD,"[{'$type': 'Group', 'FullPath': 'Group Structure\Hudson Bay Capital\AB', 'ObjectId': '148055837749805057'}]",NaN,NaN,NaN,Strategy,Credit,113434415614404864,SubStrategy,25369-AB,113997365567872256,User,AB,281474976716288,LegalEntity,JPMC,28147497671087105,CDXTranche,CDXTranche0024666,85286917943777024,LegalEntity,HBMFLTD,28147497671086592,NaN,NaN,NaN,CDXHY47V1 500 12/20/31 0-15 JPM1,2026-09-21,2031-12-20,1.0,USD,1.0,NaN,0.05,NaN,NaN,NaN,NaN,True,NaN,False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,[],NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unfunded,"{'$type': 'CDSIndex', 'Name': 'CDX.NA.HY.47-V1', 'ObjectId': '85849867896784130'}",HY Equity,25369,CDSE005141,1121331,1121331,CDX.NA.HY
1,CDXTrade,255860753830673152,CDX0024664,Done,New,2026-11-16,2026-11-16,NaN,98300000.0,0.00,NaN,0.00000,[],False,0.0,USD,"[{'$type': 'Group', 'FullPath': 'Group Structure\Hudson Bay Capital\AB', 'ObjectId': '148055837749805057'}]",NaN,NaN,NaN,Strategy,Credit,113434415614404864,SubStrategy,25369-AB,113997365567872256,User,AB,281474976716288,LegalEntity,BCI,28147497671219712,CDX,CDX-NAHYS43V5-5Y,85568392920462124,LegalEntity,HBMFLTD,28147497671086592,LegalEntity,HBMFLTD-BCI-DERCLR,28147497671201536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,HY Equity,25369,CDSE004950,1121470,1121470,CDX.NA.HY
2,CDXTrade,255860753830670107,CDX0024661,Done,New,2026-11-16,2026-11-17,NaN,98

113 columns; trader column(s): ['Trader.$type', 'Trader.Name', 'Trader.ObjectId']


## 2. Load pricers (cached) and compute PV
Pricers are cached per (calc env, as-of) in `pricer_cache/` as XML in Parquet. Only trades not yet cached are requested from DSP, so rerunning is cheap; new trades are appended.

In [5]:
import pricer_cache as pc

REFRESH = False        # True = ignore today's cache and re-pull every trade (e.g. to pick up amended trades)
RETRY_ERRORS = True    # True = re-request trades that failed on an earlier run (needed after the qncore 24.0.1 upgrade;
                       # without it build_cache silently skips every previously failed trade)

# Fetches from DSP only trades not already cached for (CALC_ENV, AS_OF); new rows are appended.
# A trade is cached only if it loads AND prices. Known failures are skipped unless RETRY_ERRORS=True.
t0 = time.time()
pc.build_cache(AS_OF, CALC_ENV, ENV, trade_ids=trade_ids, refresh=REFRESH, retry_errors=RETRY_ERRORS)

# Everything below reads from the cache, so it is identical whether the pricers were just fetched or not
pricers = pc.load_cached_pricers(AS_OF, CALC_ENV, trade_ids=trade_ids)
errors = {k: v for k, v in pc.load_cache_errors(AS_OF, CALC_ENV).items() if k in set(trade_ids)}
print(f'{len(pricers)} pricers, {len(errors)} errors in {time.time()-t0:.0f}s')
if errors:
    with pd.option_context('display.max_colwidth', None):
        display(pd.Series(errors, name='LoadError').value_counts().head(10))


9730 trades requested: 9701 cached, 0 known failures skipped, 29 to fetch
29/29 requested, 0 cached, 29 failed (22s)
Done: 0 new pricers cached, 29 not cached; file is 21.9 MB -> s:\Structured Credit\Matt Stuff\quantifi_live_risk\pricer_cache\Official\2026-11-12
9701 pricers, 29 errors in 295s


LoadError
HTTP Response code: 500, with message [{\r\n  "message": "Object reference not set to an instance of an object.",\r\n  "code": 500\r\n}] from server.    20
HTTP Response code: 400, with message [Missing Correlation Surface.] from server.                                                                         9
Name: count, dtype: int64

In [6]:
# Diagnose load errors: full messages, grouped by trade type and trader
meta = pd.DataFrame([{'TradeId': t['TradeId'], 'TradeType': t['$type'],
                      'Trader': (t.get('Trader') or {}).get('Name'),
                      'Status': t.get('TradeStatus')} for t in all_trades]).set_index('TradeId')
meta['Loaded'] = meta.index.isin(pricers.keys())
meta['Error'] = pd.Series(errors)

print(meta.groupby('TradeType')['Loaded'].agg(total='size', loaded='sum').assign(
    failed=lambda d: d.total - d.loaded).to_string())
print()
print(meta.groupby('Trader')['Loaded'].agg(total='size', loaded='sum').to_string())
print()
# Full, untruncated error text (pandas cut it to "No Trad...")
for msg, n in meta['Error'].dropna().value_counts().head(10).items():
    print(f'{n:6d}  {msg}\n')

                  total  loaded  failed
TradeType                              
BondTrade            53      53       0
CDOTrade             92      87       5
CDSTrade            935     935       0
CDXOptionTrade     1484    1484       0
CDXTrade           5530    5522       8
CDXTrancheTrade    1545    1530      15
StockOptionTrade     24      23       1
StockTrade           67      67       0

                 total  loaded
Trader                        
AB                3706    3701
Abhishek Bhutra    681     681
abhutra           5343    5319

    20  HTTP Response code: 500, with message [{
  "message": "Object reference not set to an instance of an object.",
  "code": 500
}] from server.

     9  HTTP Response code: 400, with message [Missing Correlation Surface.] from server.



In [7]:
rows = []
for tid, p in pricers.items():
    row = {'TradeId': tid, 'Pricer': type(p).__name__}
    try:
        row['Product'] = str(p.Product)
    except Exception:
        pass
    try:
        row['PV'] = p.Pv()
    except Exception as e:
        row['PvError'] = str(e)
    rows.append(row)

risk_df = pd.DataFrame(rows).set_index('TradeId')
display(risk_df.style.format({'PV': '{:,.2f}'}))
risk_df.to_csv('all_trades_pv.csv')

if errors:
    display(pd.Series(errors, name='LoadError'))

,Pricer,Product,PV
TradeId,,,
Bond_0372461,BondPricer,OBL 2.4% Apr30,0.00
Bond_0372466,BondPricer,OBL 2.4% Apr30,0.00
Bond_0382145,BondPricer,OBL 2.4% Apr30,0.00
Bond0022202,BondPricer,OBL 2.4% Apr30,0.00
Bond0022234,BondPricer,OBL 2.4% Apr30,0.00
Bond0022240,BondPricer,ATOS 5% Dec30,0.00
Bond0022271,BondPricer,ATOS 5% Dec30,0.00
Bond0022272,BondPricer,OBL 2.5% Oct29,0.00
Bond0022273,BondPricer,OBL 2.5% Oct29,0.00


CDO0017766             HTTP Response code: 500, with message [{\r\n  ...
CDO0018476             HTTP Response code: 500, with message [{\r\n  ...
CDO0018645             HTTP Response code: 500, with message [{\r\n  ...
CDO0019570             HTTP Response code: 500, with message [{\r\n  ...
CDO0022363             HTTP Response code: 500, with message [{\r\n  ...
CDX0018667             HTTP Response code: 500, with message [{\r\n  ...
CDX0019188             HTTP Response code: 500, with message [{\r\n  ...
CDX0019953             HTTP Response code: 500, with message [{\r\n  ...
CDX0024661             HTTP Response code: 500, with message [{\r\n  ...
CDX0024662             HTTP Response code: 500, with message [{\r\n  ...
CDX0024663             HTTP Response code: 500, with message [{\r\n  ...
CDX0024664             HTTP Response code: 500, with message [{\r\n  ...
CDX0024665             HTTP Response code: 500, with message [{\r\n  ...
CDXTranche0017765      HTTP Response code: 500, wit

In [8]:
# One line per trade that failed: what it is, and a short reason
import re

def short_err(msg, width=110):
    m = str(msg).strip().splitlines()[0] if str(msg).strip() else ''
    m = re.sub(r'^System\.\w+(\.\w+)*Exception:\s*', '', m)                     # drop .NET exception class prefix
    u = re.search(r'unknown field in type (\S+)', m)
    if u:                                                                     # Toolkit version mismatch
        return f"unknown field {m.split(':')[0]} in {u.group(1).split('.')[-1]} (Toolkit version?)"
    b = re.findall(r'(?:Missing|BindError) (IRisk\w+/[^;]+)', m)               # missing market object on DSP side
    if b:
        return 'BindError ' + b[-1].replace('13-Nov-2026/', '').replace('12-Nov-2026/', '')
    return m[:width]

failed = {tid: ('load', e) for tid, e in errors.items()}
failed.update({tid: ('pv', r['PvError']) for tid, r in risk_df.iterrows()
               if isinstance(r.get('PvError'), str)})

err_df = pd.DataFrame([{'TradeId': tid, 'Stage': st, 'Error': short_err(e)} for tid, (st, e) in failed.items()])
if err_df.empty:
    print('No errors')
else:
    info = {t['TradeId']: t for t in all_trades}
    err_df['Type'] = err_df.TradeId.map(lambda i: info.get(i, {}).get('$type'))
    err_df['Strategy'] = err_df.TradeId.map(lambda i: (info.get(i, {}).get('Strategy') or {}).get('Name'))
    err_df['Status'] = err_df.TradeId.map(lambda i: info.get(i, {}).get('TradeStatus'))
    err_df = err_df.sort_values(['Error', 'Type', 'TradeId'])[['TradeId', 'Type', 'Strategy', 'Status', 'Stage', 'Error']]
    print(f'{len(err_df)} trades with errors')
    display(err_df.Error.value_counts().rename('trades').to_frame())
    with pd.option_context('display.max_rows', None, 'display.max_colwidth', 130, 'display.width', 250):
        display(err_df.reset_index(drop=True))


29 trades with errors


,trades
Error,
"HTTP Response code: 500, with message [{",20
"HTTP Response code: 400, with message [Missing Correlation Surface.] from server.",9


,TradeId,Type,Strategy,Status,Stage,Error
0,CDXTranche0019211,CDXTrancheTrade,CV Draft Trades,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
1,CDXTranche0019212,CDXTrancheTrade,CV Draft Trades,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
2,CDXTranche0022093,CDXTrancheTrade,TradeIdeas,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
3,CDXTranche0022107,CDXTrancheTrade,TradeIdeas,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
4,CDXTranche0022622,CDXTrancheTrade,TradeIdeas,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
5,CDXTranche0024660,CDXTrancheTrade,Manual,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
6,CDXTranche0024666,CDXTrancheTrade,Credit,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
7,CDXTranche0024667,CDXTrancheTrade,Credit,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
8,StockOption_0784563,StockOptionTrade,Credit,Done,load,"HTTP Response code: 400, with message [Missing Correlation Surface.] from server."
9,CDO0017766,CDOTrade,TradeIdeas,Pending,load,"HTTP Response code: 500, with message [{"
